## 1. My Rule and Signal Checks

My baseline rule will prioritize pages that already receive meaningful search visibility but have weaker-than-expected CTR for their average search position.

Before writing the rule, I check two signals:

1. **CTR versus position:** Pages ranking in better positions should generally receive higher CTR.
2. **Search volume:** Pages with more impressions represent larger potential opportunities, although impressions alone do not prove that a page needs fixing.

The checks use March 2026 only. All features are calculated from information available during that month, with no future-window or outcome-derived inputs.

In [8]:
import pandas as pd

table_path = (
    f"{rel}/fact_content_daily_performance/**/*.parquet"
)

# Aggregate daily rows into one row per client-content page for March 2026.
march_pages = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS avg_position
FROM read_parquet('{table_path}')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

print("Monthly page rows:", len(march_pages))
display(march_pages.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly page rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,2.298246
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542


In [9]:
# Signal 1: CTR versus average search position

position_table = con.sql(f"""
WITH pages AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
),
bucketed AS (
    SELECT *,
        CASE
            WHEN avg_position <= 3 THEN '01: positions 1-3'
            WHEN avg_position <= 10 THEN '02: positions 4-10'
            WHEN avg_position <= 20 THEN '03: positions 11-20'
            ELSE '04: position 21+'
        END AS position_bucket
    FROM pages
)
SELECT
    position_bucket,
    COUNT(*) AS n,
    SUM(impressions) AS impressions,
    SUM(clicks) AS clicks,
    SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS weighted_ctr
FROM bucketed
GROUP BY position_bucket
ORDER BY position_bucket
""").df()

print("Signal 1: CTR versus position")
display(position_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR versus position


,position_bucket,n,impressions,clicks,weighted_ctr
0,01: positions 1-3,18860,41420140.0,160562.0,0.003876
1,02: positions 4-10,83288,148112436.0,481189.0,0.003249
2,03: positions 11-20,29922,31191659.0,98488.0,0.003158
3,04: position 21+,44668,59933354.0,81593.0,0.001361


In [10]:
# Signal 2: Search volume through monthly impressions

volume_table = con.sql(f"""
WITH pages AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
),
bucketed AS (
    SELECT *,
        CASE
            WHEN impressions < 10 THEN '01: 1-9'
            WHEN impressions < 100 THEN '02: 10-99'
            WHEN impressions < 1000 THEN '03: 100-999'
            ELSE '04: 1000+'
        END AS impression_bucket
    FROM pages
)
SELECT
    impression_bucket,
    COUNT(*) AS n,
    SUM(impressions) AS impressions,
    SUM(clicks) AS clicks,
    SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS weighted_ctr
FROM bucketed
GROUP BY impression_bucket
ORDER BY impression_bucket
""").df()

print("Signal 2: Search volume")
display(volume_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2: Search volume


,impression_bucket,n,impressions,clicks,weighted_ctr
0,01: 1-9,33532,114683.0,1093.0,0.009531
1,02: 10-99,41765,1746823.0,5364.0,0.003071
2,03: 100-999,56383,21967732.0,53028.0,0.002414
3,04: 1000+,45058,256828351.0,762347.0,0.002968


## Signal Checks

### Signal 1: CTR vs Average Position
**Verdict:** CONFIRMED

Weighted CTR decreases as average search position becomes worse. Pages in positions 1–3 have the highest weighted CTR (0.3876%), while pages beyond position 20 have the lowest (0.1361%). This supports using CTR relative to position as part of the baseline rule.

### Signal 2: Search Volume (Impressions)
**Verdict:** MIXED

Higher-impression pages represent larger opportunities, but impressions alone do not consistently correspond to higher CTR. Very low-impression pages show the highest weighted CTR, likely because of small sample sizes. Impressions are therefore useful for prioritization but should not be used as the only decision signal.

In [11]:
import os

baseline = march_pages.copy()

# Expected CTR threshold based on position
baseline["expected_ctr"] = baseline["avg_position"].apply(
    lambda x: 0.004 if x <= 10 else 0.002
)

baseline["ctr_gap"] = (
    baseline["expected_ctr"] - baseline["ctr"]
).clip(lower=0)

baseline["score"] = (
    baseline["impressions"] * baseline["ctr_gap"]
)

baseline["reason_code"] = "LOW_CTR_VISIBLE_PAGE"

baseline["action"] = baseline["score"].apply(
    lambda s: "Review CTR & Content" if s > 0 else "Monitor"
)

baseline = baseline.sort_values(
    "score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(baseline))
print()

display(
    baseline[
        [
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Rows: 176738



,content_hash_id,impressions,ctr,avg_position,score,reason_code,action
18642,content_44f34c0a90047651,212404.0,0.000113,0.665877,825.616,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
88850,content_8e1334d6356668e3,134984.0,0.000007,2.693038,538.936,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
7624,content_34a70fea29d15f24,143019.0,0.000301,3.166132,529.076,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
125669,content_8d7d99f109e19aa2,203497.0,0.001420,2.468557,524.988,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
281,content_fec55986a1868d62,124075.0,0.000008,0.308426,495.300,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
40651,content_7c6373141eae744a,132593.0,0.000626,5.948459,447.372,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
40680,content_acbcc847f8996314,170808.0,0.001534,3.396293,421.232,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
129197,content_b99ea6861864dea5,194337.0,0.001858,4.551516,416.348,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
118384,content_f6116743b00afc2d,107584.0,0.000139,9.735658,415.336,LOW_CTR_VISIBLE_PAGE,Review CTR & Content
132773,content_f43118e089ecc69a,139417.0,0.001370,5.342218,366.668,LOW_CTR_VISIBLE_PAGE,Review CTR & Content


## Top-20 Review

| Rank | Action | Reason Code | Confidence | What would make it wrong |
|------|--------|-------------|------------|--------------------------|
| 1 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | High | CTR may be affected by branded searches or SERP features rather than page quality. |
| 2 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | High | High impressions alone do not guarantee that improving the page will increase clicks. |
| 3 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | High | Search intent may not match the page content despite strong visibility. |
| 4 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | High | Seasonal demand or temporary ranking changes could explain the low CTR. |
| 5 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | High | Rich search features may reduce clicks even with strong rankings. |
| 6 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Average position may hide day-to-day ranking variation. |
| 7 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | The page may already be optimized, with external factors limiting CTR. |
| 8 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Aggregated monthly data may hide short-term improvements. |
| 9 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Search Console measurements may contain natural variability. |
| 10 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Some impressions may come from irrelevant queries. |
| 11 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Query mix may differ from other pages with similar rankings. |
| 12 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Position alone cannot explain all click behaviour. |
| 13 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | User behaviour may change across devices or regions. |
| 14 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Low CTR may not be actionable without query-level analysis. |
| 15 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Monthly aggregation may smooth important daily changes. |
| 16 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Some pages naturally receive fewer clicks despite good rankings. |
| 17 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | External events may temporarily affect performance. |
| 18 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | The rule does not consider content freshness. |
| 19 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Search demand may fluctuate outside this month. |
| 20 | Review CTR & Content | LOW_CTR_VISIBLE_PAGE | Medium | Additional features could reorder these pages. |

## Weak Picks + Leakage Check

Some highly ranked pages may be false positives because the baseline rule only considers impressions, average position, and CTR. It does not account for query intent, branded searches, SERP features, or seasonality.

No future-window information was used. The score was calculated only from March 2026 observations, and no product flags, labels, or manually created outcomes were used in building the ranking.

In [12]:
import os

baseline = march_pages.copy()

# Expected CTR based on average position
baseline["expected_ctr"] = baseline["avg_position"].apply(
    lambda x: 0.004 if x <= 10 else 0.002
)

baseline["ctr_gap"] = (
    baseline["expected_ctr"] - baseline["ctr"]
).clip(lower=0)

# Baseline score
baseline["score"] = (
    baseline["impressions"] * baseline["ctr_gap"]
)

# One reason code per page
def reason(row):
    if row["avg_position"] <= 3:
        return "LOW_CTR_TOP_POSITION"
    elif row["impressions"] >= 10000:
        return "LOW_CTR_HIGH_IMPRESSIONS"
    else:
        return "LOW_CTR_VISIBLE_PAGE"

baseline["reason_code"] = baseline.apply(reason, axis=1)

# One action label
baseline["action"] = baseline["score"].apply(
    lambda s: "Review CTR & Content" if s > 0 else "Monitor"
)

baseline = baseline.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(baseline))
print()

display(
    baseline[
        [
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "score",
            "reason_code",
            "action",
        ]
    ].head(20)
)

Rows: 176738



,content_hash_id,impressions,ctr,avg_position,score,reason_code,action
18642,content_44f34c0a90047651,212404.0,0.000113,0.665877,825.616,LOW_CTR_TOP_POSITION,Review CTR & Content
88850,content_8e1334d6356668e3,134984.0,0.000007,2.693038,538.936,LOW_CTR_TOP_POSITION,Review CTR & Content
7624,content_34a70fea29d15f24,143019.0,0.000301,3.166132,529.076,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content
125669,content_8d7d99f109e19aa2,203497.0,0.001420,2.468557,524.988,LOW_CTR_TOP_POSITION,Review CTR & Content
281,content_fec55986a1868d62,124075.0,0.000008,0.308426,495.300,LOW_CTR_TOP_POSITION,Review CTR & Content
40651,content_7c6373141eae744a,132593.0,0.000626,5.948459,447.372,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content
40680,content_acbcc847f8996314,170808.0,0.001534,3.396293,421.232,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content
129197,content_b99ea6861864dea5,194337.0,0.001858,4.551516,416.348,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content
118384,content_f6116743b00afc2d,107584.0,0.000139,9.735658,415.336,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content
132773,content_f43118e089ecc69a,139417.0,0.001370,5.342218,366.668,LOW_CTR_HIGH_IMPRESSIONS,Review CTR & Content


## My Rule and Reason Codes

### Rule

Rank pages that receive meaningful search visibility but have a lower-than-expected CTR for their average search position. Higher impressions increase priority because improving those pages could affect more users.

### Reason Codes

- LOW_CTR_TOP_POSITION – Page ranks in the top 3 but CTR is lower than expected.
- LOW_CTR_HIGH_IMPRESSIONS – Page receives high search visibility but CTR is below expectation.
- LOW_CTR_VISIBLE_PAGE – Page has a measurable CTR gap and should be reviewed.

### Action Label

- Review CTR & Content
- Monitor